# Invertible trajectory-encoding mapping

Train an invertible NSF whose forward map sends an intended target encoding $z'$ to the policy input $z$. Training evaluates the reverse map on buffer samples.

For a replay sample $(z, \log q(z), y, s)$, the reverse image is $z'=R_\theta(z)$ and the per-trajectory discrepancy is

$$d_\theta = \operatorname{mean}_{t,xy}\left[(y-Az')^2s + 25(1-s)\right].$$

The pullback of $p$ through the reverse map is

$$\log q'_\theta(z)=\log p(R_\theta(z))+\log|\det J_{R_\theta}(z)|,$$

so replay samples from $q$ use $w=\exp(\log q'_\theta-\log q)$. The regularizer samples $z'\sim p$ and applies the forward map $z=F_\theta(z')$:

$$D_{KL}(q'_\theta\|q)=\mathbb{E}_{z'\sim p}\left[\log p(z')-\log|\det J_{F_\theta}(z')|-\log q(z)\right].$$

For this initial mechanism test, the same frozen normalizing flow supplies both $p$ and $q$.

The optimized loss mixes the reweighted discrepancy and KL according to the normalized effective sample size, with the ESS fraction detached from the gradient:

$$\mathcal L=\operatorname{ESS}\cdot\operatorname{MSE}+(1-\operatorname{ESS})\cdot D_{KL}(q'_\theta\|q).$$

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

from flax import serialization
from flax.core import freeze
from flax.training.train_state import TrainState
import jax
import jax.numpy as jnp
import numpy as np
import optax

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jax_rq_nsf").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from jax_rq_nsf import FlowConfig, init_flow, init_invertible_nsf

SEED = 8848
BATCH_SIZE = int(os.environ.get("TBQDRL_MAPPING_BATCH_SIZE", "4096"))
TRAINING_STEPS = int(os.environ.get("TBQDRL_MAPPING_TRAINING_STEPS", "5000"))
REPORT_PERIOD = int(os.environ.get("TBQDRL_MAPPING_REPORT_PERIOD", "100"))
LEARNING_RATE = 3e-4
DEATH_ERROR = 10.0

TRAJECTORY_BUFFER_PATH = Path(
    os.environ.get(
        "TBQDRL_TRAJECTORY_BUFFER",
        PROJECT_ROOT / "output/mapping_test/trajectory_buffer.npz",
    )
)
PROJECTION_MATRIX_PATH = (
    PROJECT_ROOT / "output/mapping_test/trajectory_projection_matrix.npy"
)
FLOW_CHECKPOINT_PATH = (
    PROJECT_ROOT / "output/vi_filtered/trajectory_flow_variables.msgpack"
)
OUTPUT_DIRECTORY = Path(
    os.environ.get(
        "TBQDRL_MAPPING_OUTPUT_DIRECTORY",
        PROJECT_ROOT / "output/mapping_test",
    )
)
MAPPING_CHECKPOINT_PATH = OUTPUT_DIRECTORY / "mapping_variables.msgpack"
TRAINING_HISTORY_PATH = OUTPUT_DIRECTORY / "mapping_training_history.npz"
METADATA_PATH = OUTPUT_DIRECTORY / "mapping_metadata.json"

FLOW_CONFIG = FlowConfig(
    dimension=50,
    num_layers=8,
    hidden_features=(128, 128),
    num_bins=16,
    tail_bound=6.0,
)
MAPPING_CONFIG = FlowConfig(
    dimension=50,
    num_layers=4,
    hidden_features=(128, 128),
    num_bins=16,
    tail_bound=4.0,
)

assert BATCH_SIZE > 0
assert TRAINING_STEPS > 0
assert REPORT_PERIOD > 0
print("JAX devices:", jax.devices())
print("Batch size:", BATCH_SIZE)
print("Training steps:", TRAINING_STEPS)
print("Loss: ESS * MSE + (1 - ESS) * KL")

In [ ]:
assert TRAJECTORY_BUFFER_PATH.is_file(), TRAJECTORY_BUFFER_PATH
assert PROJECTION_MATRIX_PATH.is_file(), PROJECTION_MATRIX_PATH
assert FLOW_CHECKPOINT_PATH.is_file(), FLOW_CHECKPOINT_PATH

with np.load(TRAJECTORY_BUFFER_PATH, allow_pickle=False) as archive:
    buffer_z = archive["z"]
    buffer_log_q = archive["log_q"]
    buffer_xy = archive["xy"]
    buffer_scale = archive["scale"]

projection_matrix = np.load(PROJECTION_MATRIX_PATH, allow_pickle=False)
projection_matrix = jnp.asarray(projection_matrix, dtype=jnp.float32)

num_samples = buffer_z.shape[0]
assert buffer_z.shape == (num_samples, 50)
assert buffer_log_q.shape == (num_samples, 1)
assert buffer_xy.shape == (num_samples, 96, 2)
assert buffer_scale.shape == (num_samples, 96, 1)
assert projection_matrix.shape == (96, 25)
assert num_samples >= BATCH_SIZE

buffer_size_gib = sum(
    array.nbytes
    for array in (buffer_z, buffer_log_q, buffer_xy, buffer_scale)
) / 1024**3
print("Trajectory samples:", f"{num_samples:,}")
print("Loaded buffer size:", f"{buffer_size_gib:.2f} GiB")
print("Projection matrix:", projection_matrix.shape, projection_matrix.dtype)

In [ ]:
key = jax.random.PRNGKey(SEED)
key, flow_init_key, mapping_init_key, training_key = jax.random.split(key, 4)

flow, flow_variables_template = init_flow(flow_init_key, FLOW_CONFIG)
flow_variables = serialization.from_bytes(
    flow_variables_template,
    FLOW_CHECKPOINT_PATH.read_bytes(),
)

mapping, mapping_variables = init_invertible_nsf(
    mapping_init_key,
    MAPPING_CONFIG,
)
training_state = TrainState.create(
    apply_fn=mapping.apply,
    params=mapping_variables["params"],
    tx=optax.adam(LEARNING_RATE),
)

mapping_parameter_count = sum(
    leaf.size for leaf in jax.tree.leaves(training_state.params)
)
print("Flow checkpoint:", FLOW_CHECKPOINT_PATH)
print("Mapping parameters:", f"{mapping_parameter_count:,}")

In [ ]:
def decode_trajectory(static_z):
    """Apply the shared (96, 25) matrix to both encoded axes."""
    per_axis_z = static_z.reshape(static_z.shape[:-1] + (2, 25))
    return jnp.einsum("td,...cd->...tc", projection_matrix, per_axis_z)


def loss_function(params, batch, p_sample_key):
    z, stored_log_q, xy, scale = batch

    mapped_z, reverse_log_det = mapping.apply(
        {"params": params},
        z,
        method=mapping.reverse,
    )
    target_xy = decode_trajectory(mapped_z)
    per_sample_discrepancy = jnp.mean(
        jnp.square(xy - target_xy) * scale
        + (1.0 - scale) * DEATH_ERROR,
        axis=(-1, -2),
    )

    # q'_theta is the pullback of p through the reverse map. The buffer was
    # sampled from q, so this ratio changes the discrepancy expectation to q'.
    mapped_log_p = flow.apply(
        flow_variables,
        mapped_z,
        method=flow.log_prob,
    )
    log_importance_ratio = (
        mapped_log_p + reverse_log_det - stored_log_q[..., 0]
    )
    importance_ratio = jnp.exp(log_importance_ratio)
    discrepancy_loss = jnp.mean(
        importance_ratio * per_sample_discrepancy
    )

    # Independently sample z' ~ p and evaluate KL(q'_theta || q) after
    # the forward map z = F_theta(z').
    p_samples = flow.apply(
        flow_variables,
        p_sample_key,
        BATCH_SIZE,
        method=flow.sample,
    )
    p_log_prob = flow.apply(
        flow_variables,
        p_samples,
        method=flow.log_prob,
    )
    projected_z, forward_log_det = mapping.apply(
        {"params": params},
        p_samples,
        method=mapping.forward,
    )
    projected_log_prob = p_log_prob - forward_log_det
    q_log_prob = flow.apply(
        flow_variables,
        projected_z,
        method=flow.log_prob,
    )
    kl_divergence = jnp.mean(projected_log_prob - q_log_prob)

    effective_sample_size = (
        jnp.square(jnp.sum(importance_ratio))
        / (jnp.sum(jnp.square(importance_ratio)) + 1e-8)
        / BATCH_SIZE
    )
    kl_loss = kl_divergence
    mse_loss = discrepancy_loss
    ess = jax.lax.stop_gradient(
        jnp.clip(effective_sample_size, 0.0, 1.0)
    )
    total_loss = ess * mse_loss + (1.0 - ess) * kl_loss
    metrics = {
        "total_loss": total_loss,
        "mse_loss": mse_loss,
        "unweighted_mse": jnp.mean(per_sample_discrepancy),
        "kl_divergence": kl_loss,
        "importance_mean": jnp.mean(importance_ratio),
        "importance_max": jnp.max(importance_ratio),
        "ess_fraction": effective_sample_size,
    }
    return total_loss, metrics


@jax.jit
def train_step(state, batch, p_sample_key):
    (_, metrics), gradients = jax.value_and_grad(
        loss_function,
        has_aux=True,
    )(state.params, batch, p_sample_key)
    state = state.apply_gradients(grads=gradients)
    return state, metrics

In [ ]:
class ShuffledTrajectorySampler:
    """Draw non-repeating shuffled batches, reshuffling at each boundary."""

    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)
        self.order = self.rng.permutation(num_samples)
        self.offset = 0

    def next_batch(self):
        if self.offset + BATCH_SIZE > num_samples:
            self.order = self.rng.permutation(num_samples)
            self.offset = 0
        indices = self.order[self.offset : self.offset + BATCH_SIZE]
        self.offset += BATCH_SIZE
        return (
            buffer_z[indices],
            buffer_log_q[indices],
            buffer_xy[indices],
            buffer_scale[indices],
        )


trajectory_sampler = ShuffledTrajectorySampler(SEED)
print("Complete batches per shuffle:", num_samples // BATCH_SIZE)

In [ ]:
history = {
    "step": [],
    "total_loss": [],
    "mse_loss": [],
    "unweighted_mse": [],
    "kl_divergence": [],
    "importance_mean": [],
    "importance_max": [],
    "ess_fraction": [],
}

start_time = time.monotonic()
for step in range(1, TRAINING_STEPS + 1):
    training_key, p_sample_key = jax.random.split(training_key)
    batch = trajectory_sampler.next_batch()
    training_state, metrics = train_step(
        training_state,
        batch,
        p_sample_key,
    )

    if step == 1 or step % REPORT_PERIOD == 0 or step == TRAINING_STEPS:
        host_metrics = jax.device_get(metrics)
        if not np.isfinite(float(host_metrics["total_loss"])):
            raise FloatingPointError(f"non-finite loss at step {step}")
        history["step"].append(step)
        for name in history:
            if name != "step":
                history[name].append(float(host_metrics[name]))

        elapsed = time.monotonic() - start_time
        ess = float(np.clip(host_metrics["ess_fraction"], 0.0, 1.0))
        mse = float(host_metrics["mse_loss"])
        kl = float(host_metrics["kl_divergence"])
        print(
            f"step {step:6d}/{TRAINING_STEPS} | "
            f"MSE {mse:.6f} | "
            f"KL {kl:.6f} | "
            f"total {ess:.3f}*MSE + {1.0 - ess:.3f}*KL "
            f"= {float(host_metrics['total_loss']):.6f} | "
            f"ESS {ess:.3f} | "
            f"{elapsed:.1f}s"
        )

In [ ]:
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
trained_mapping_variables = freeze({"params": training_state.params})
MAPPING_CHECKPOINT_PATH.write_bytes(
    serialization.to_bytes(trained_mapping_variables)
)
np.savez(
    TRAINING_HISTORY_PATH,
    **{
        name: np.asarray(values)
        for name, values in history.items()
    },
)
metadata = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "training_steps": TRAINING_STEPS,
    "learning_rate": LEARNING_RATE,
    "loss": "stopgrad(ESS) * MSE + (1 - stopgrad(ESS)) * KL",
    "death_error": DEATH_ERROR,
    "trajectory_buffer": str(TRAJECTORY_BUFFER_PATH),
    "projection_matrix": str(PROJECTION_MATRIX_PATH),
    "flow_checkpoint_for_p_and_q": str(FLOW_CHECKPOINT_PATH),
    "mapping_dimension": MAPPING_CONFIG.dimension,
    "mapping_num_layers": MAPPING_CONFIG.num_layers,
    "mapping_hidden_features": list(MAPPING_CONFIG.hidden_features),
    "mapping_num_bins": MAPPING_CONFIG.num_bins,
    "mapping_tail_bound": MAPPING_CONFIG.tail_bound,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2) + "\n")

print("Saved mapping:", MAPPING_CHECKPOINT_PATH)
print("Saved history:", TRAINING_HISTORY_PATH)
print("Saved metadata:", METADATA_PATH)